 Three specifications per model:
   1. Spec A — no school variables (baseline)
   2. Spec B — binary treatment (near_tier1_1km)
   3. Spec C — continuous treatment (nearest_tier1_primary_school_dist_m)

Models:
   1. OLS       — naive benchmark, interpretable coefficients
   2. Ridge     — handles multicollinearity, no feature selection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.api as sm
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score

import warnings
warnings.filterwarnings("ignore")

df_ols = pd.read_csv("../cleaned_datasets/df_ols.csv")

target       = "log_resale_price"
d_continuous = "nearest_tier1_primary_school_dist_m"
d_binary     = "near_tier1_1km"

base_features = [
    c for c in df_ols.columns
    if c not in [target, d_binary, d_continuous]
]
print(base_features)

# Spec A — no school variables
features_A = base_features

# Spec B — with binary treatment
features_B = base_features + [d_binary]

# Spec C — with continuous treatment
features_C = base_features + [d_continuous]

y = df_ols[target]

specs = {
    "Spec A — no school vars" : features_A,
    "Spec B — binary treatment"     : features_B,
    "Spec C — continuous treatment" : features_C,
}

['floor_area_sqm', 'num_schools_1km', 'num_schools_2km', 'num_tier1_schools_1km', 'num_tier1_schools_2km', 'num_tier2_schools_1km', 'num_tier2_schools_2km', 'num_unique_mrt_500m', 'num_unique_mrt_1km', 'num_unique_mrt_lines_500m', 'num_unique_mrt_lines_1km', 'walking_dist_mrt_m', 'num_busstops_500m', 'num_busstops_1km', 'walking_dist_busstop_m', 'num_unique_hawker_500m', 'num_unique_hawker_1km', 'walking_dist_hawker_m', 'num_malls_1km', 'num_malls_2km', 'walking_dist_mall_m', 'dist_cbd_m', 'town_BEDOK', 'town_BISHAN', 'town_BUKIT BATOK', 'town_BUKIT MERAH', 'town_BUKIT PANJANG', 'town_BUKIT TIMAH', 'town_CENTRAL AREA', 'town_CHOA CHU KANG', 'town_CLEMENTI', 'town_GEYLANG', 'town_HOUGANG', 'town_JURONG EAST', 'town_JURONG WEST', 'town_KALLANG/WHAMPOA', 'town_MARINE PARADE', 'town_PASIR RIS', 'town_PUNGGOL', 'town_QUEENSTOWN', 'town_SEMBAWANG', 'town_SENGKANG', 'town_SERANGOON', 'town_TAMPINES', 'town_TOA PAYOH', 'town_WOODLANDS', 'town_YISHUN', 'flat_type_2 ROOM', 'flat_type_3 ROOM', 'f

In [ ]:
#OLS

mean_price = np.exp(y.mean())

# Spec A — no school vars
ols_A     = sm.OLS(y, sm.add_constant(df_ols[features_A])).fit(cov_type="HC3")
r2_ols_A  = ols_A.rsquared
adj_r2_ols_A = ols_A.rsquared_adj

print(f"Spec A | R²={r2_ols_A:.4f} | Adj R²={adj_r2_ols_A:.4f}")

# Spec B — binary treatment
ols_B     = sm.OLS(y, sm.add_constant(df_ols[features_B])).fit(cov_type="HC3")
r2_ols_B  = ols_B.rsquared
adj_r2_ols_B = ols_B.rsquared_adj

coef_B  = ols_B.params[d_binary]
pval_B  = ols_B.pvalues[d_binary]
ci_B    = ols_B.conf_int().loc[d_binary]
sgd_B   = (np.exp(coef_B) - 1) * mean_price
sig_B   = "***" if pval_B < 0.01 else ("**" if pval_B < 0.05 else ("*" if pval_B < 0.10 else "n.s."))

print(f"Spec B | R²={r2_ols_B:.4f} | Adj R²={adj_r2_ols_B:.4f}")
print(f"       | {d_binary}: β={coef_B:.6f} ({sig_B}) p={pval_B:.4f}")
print(f"       | 95% CI=[{ci_B[0]:.6f}, {ci_B[1]:.6f}]")
print(f"       | SGD at mean price = SGD {sgd_B:,.0f}")

# Spec C — continuous treatment
ols_C     = sm.OLS(y, sm.add_constant(df_ols[features_C])).fit(cov_type="HC3")
r2_ols_C  = ols_C.rsquared
adj_r2_ols_C = ols_C.rsquared_adj

coef_C  = ols_C.params[d_continuous]
pval_C  = ols_C.pvalues[d_continuous]
ci_C    = ols_C.conf_int().loc[d_continuous]
sgd_C   = (np.exp(coef_C) - 1) * mean_price
sig_C   = "***" if pval_C < 0.01 else ("**" if pval_C < 0.05 else ("*" if pval_C < 0.10 else "n.s."))

print(f"Spec C | R²={r2_ols_C:.4f} | Adj R²={adj_r2_ols_C:.4f}")
print(f"       | {d_continuous}: β={coef_C:.6f} ({sig_C}) p={pval_C:.4f}")
print(f"       | 95% CI=[{ci_C[0]:.6f}, {ci_C[1]:.6f}]")
print(f"       | SGD at mean price = SGD {sgd_C:,.2f}")

print(f"\nOLS ΔR²:")
print(f"  Spec A : {r2_ols_A:.6f}")
print(f"  Spec B : {r2_ols_B:.6f}  ΔR² = {r2_ols_B - r2_ols_A:+.6f}")
print(f"  Spec C : {r2_ols_C:.6f}  ΔR² = {r2_ols_C - r2_ols_A:+.6f}")


# store for summary table
ols_results = {
    "Spec A — no school vars"      : ols_A,
    "Spec B — binary treatment"    : ols_B,
    "Spec C — continuous treatment": ols_C,
}

Spec A | R²=0.9298 | Adj R²=0.9297
Spec B | R²=0.9298 | Adj R²=0.9298
       | near_tier1_1km: β=0.027823 (***) p=0.0000
       | 95% CI=[0.023584, 0.032062]
       | SGD at mean price = SGD 17,969
Spec C | R²=0.9298 | Adj R²=0.9297
       | nearest_tier1_primary_school_dist_m: β=0.000002 (***) p=0.0003
       | 95% CI=[0.000001, 0.000003]
       | SGD at mean price = SGD 1.15

OLS ΔR²:
  Spec A : 0.929754
  Spec B : 0.929827  ΔR² = +0.000073
  Spec C : 0.929760  ΔR² = +0.000007

*** p<0.01  ** p<0.05  * p<0.10  n.s. not significant
Standard errors: HC3 heteroskedasticity-robust


In [ ]:
# Standardising features for penalised regression

scalers   = {}
X_scaled  = {}

for spec_label, feat_cols in specs.items():
    scaler = StandardScaler()
    X_sc   = pd.DataFrame(
        scaler.fit_transform(df_ols[feat_cols]),
        columns = feat_cols,
        index   = df_ols.index,
    )
    scalers[spec_label]  = scaler
    X_scaled[spec_label] = X_sc

In [ ]:
# RIDGE REGRESSION
alphas_ridge = np.logspace(-3, 6, 30)
kf3 = KFold(n_splits=3, shuffle=True, random_state=42)

# Spec A — no school vars
ridge_A = RidgeCV(alphas=alphas_ridge, cv=kf3, scoring="r2")
ridge_A.fit(X_scaled["Spec A — no school vars"], y)
coefs_ridge_A = pd.Series(ridge_A.coef_, index=features_A)
r2_ridge_A    = ridge_A.best_score_

print(f"Spec A | alpha={ridge_A.alpha_:.4f} | CV R²={r2_ridge_A:.4f}")

# Spec B — binary treatment
ridge_B = RidgeCV(alphas=alphas_ridge, cv=kf3, scoring="r2")
ridge_B.fit(X_scaled["Spec B — binary treatment"], y)
coefs_ridge_B = pd.Series(ridge_B.coef_, index=features_B)
r2_ridge_B    = ridge_B.best_score_

print(f"Spec B | alpha={ridge_B.alpha_:.4f} | CV R²={r2_ridge_B:.4f}")
print(f"       | {d_binary} coef={coefs_ridge_B[d_binary]:.6f}")

# Spec C — continuous treatment
ridge_C = RidgeCV(alphas=alphas_ridge, cv=kf3, scoring="r2")
ridge_C.fit(X_scaled["Spec C — continuous treatment"], y)
coefs_ridge_C = pd.Series(ridge_C.coef_, index=features_C)
r2_ridge_C    = ridge_C.best_score_

print(f"Spec C | alpha={ridge_C.alpha_:.4f} | CV R²={r2_ridge_C:.4f}")
print(f"       | {d_continuous} coef={coefs_ridge_C[d_continuous]:.6f}")

print(f"\nRidge ΔR²:")
print(f"  Spec A : {r2_ridge_A:.6f}")
print(f"  Spec B : {r2_ridge_B:.6f}  ΔR² = {r2_ridge_B - r2_ridge_A:+.6f}")
print(f"  Spec C : {r2_ridge_C:.6f}  ΔR² = {r2_ridge_C - r2_ridge_A:+.6f}")

# store for summary table
ridge_results = {
    "Spec A — no school vars"      : {"cv_r2": r2_ridge_A, "coefs": coefs_ridge_A},
    "Spec B — binary treatment"    : {"cv_r2": r2_ridge_B, "coefs": coefs_ridge_B},
    "Spec C — continuous treatment": {"cv_r2": r2_ridge_C, "coefs": coefs_ridge_C},
}

Spec A | alpha=0.6210 | CV R²=0.9297
Spec B | alpha=0.6210 | CV R²=0.9297
       | near_tier1_1km coef=0.010748
Spec C | alpha=1.2690 | CV R²=0.9297
       | nearest_tier1_primary_school_dist_m coef=0.004720

Ridge ΔR²:
  Spec A : 0.929659
  Spec B : 0.929732  ΔR² = +0.000073
  Spec C : 0.929664  ΔR² = +0.000005
